# Efecto fotoeléctrico: cuánto importa el filtro

Se mide el potencial de frenado para cinco frecuencias de luz incidente, en tres
configuraciones ópticas: sin filtro, con filtro, y con filtro Pasco. De la
pendiente de potencial contra frecuencia sale `h/e`, y de ahí la constante de
Planck.

**El punto del análisis** es que la configuración óptica domina el resultado:

| serie | h obtenido | desvío | R² |
|---|---|---|---|
| orden 1, filtro Pasco | 6,587×10⁻³⁴ J·s | −0,6% | 0,9976 |
| orden 1, filtro | 6,445×10⁻³⁴ | −2,7% | 0,9969 |
| orden 2, filtro | 6,310×10⁻³⁴ | −4,8% | 0,9613 |
| orden 1, sin filtro | 4,523×10⁻³⁴ | −31,7% | 0,9815 |
| orden 2, sin filtro | 3,230×10⁻³⁴ | −51,3% | 0,8756 |

Sin filtro el error es del 30 al 50%: entra luz de otros órdenes de difracción y
contamina la medición. Con el filtro adecuado se recupera el valor aceptado
(6,626×10⁻³⁴ J·s) con menos de 1% de desvío.

**Detalle metodológico:** la incerteza de cada tensión no se asume constante, se
modela según la especificación del instrumento, `σ_V = 0,5%·V + resolución`, y se
pasa al ajuste con `absolute_sigma=True`.

*Universidad Nacional de La Plata · Licenciatura en Física*
*Mediciones tomadas junto a Belén Robiglio.*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =======================
# comparación sin filtro - filtro - filtro pasco
# =======================
#orden 2 (sin filtro)
v1 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm1 = np.array([1.66, 1.29, 1.15, 1.05, 0.91])    # V
sigma_Vm1 = (0.5*Vm1)/100 + 0.01

#orden 2 (filtro)
v2 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm2 = np.array([1.652, 1.295, 1.030, 0.722, 0.379])    # V
sigma_Vm2 = (0.5*Vm2)/100 + 0.001

#orden 2 (filtro pasco)
v3 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm3 = np.array([1.654, 1.286, 1.010, 0.383, 0.633])    # V ----
sigma_Vm3 = (0.5*Vm3)/100 + 0.001

# =======================
# FUNCIÓN DE AJUSTE
# =======================
def lin_func(x, m, C):
    return m*x + C   # m = h/e

def ajustar(v, Vm, sigma_Vm):
    """Ajuste lineal con curve_fit y cálculo de R^2"""
    popt, pcov = curve_fit(lin_func, v, Vm, sigma=sigma_Vm,
                           absolute_sigma=True, p0=[4e-15, 0])
    m, C = popt
    sigma_m, sigma_C = np.sqrt(np.diag(pcov))
    Vm_pred = lin_func(v, *popt)
    ss_res = np.sum((Vm - Vm_pred)**2)
    ss_tot = np.sum((Vm - np.mean(Vm))**2)
    r2 = 1 - ss_res/ss_tot
    return m, C, sigma_m, sigma_C, r2

# =======================
# AJUSTES
# =======================
m1, C1, sm1, sC1, r21 = ajustar(v1, Vm1, sigma_Vm1)
m2, C2, sm2, sC2, r22 = ajustar(v2, Vm2, sigma_Vm2)
m3, C3, sm3, sC3, r23 = ajustar(v3, Vm3, sigma_Vm3)

# =======================
# RESULTADOS POR CONSOLA
# =======================
print("=== Serie 1 ===")
print(f"h/e = {m1:.3e} ± {sm1:.3e} V·s | C = {C1:.3e} ± {sC1:.3e} V | R² = {r21:.4f}")
print("\n=== Serie 2 ===")
print(f"h/e = {m2:.3e} ± {sm2:.3e} V·s | C = {C2:.3e} ± {sC2:.3e} V | R² = {r22:.4f}")
print("\n=== Serie 3 ===")
print(f"h/e = {m3:.3e} ± {sm3:.3e} V·s | C = {C3:.3e} ± {sC3:.3e} V | R² = {r23:.4f}")

# =======================
# GRÁFICO
# =======================
plt.figure(figsize=(8,6))

# Colores y símbolos diferentes
colores = ['tab:blue', 'tab:red', 'tab:green']
marcadores = ['o', 's', 'D']  # círculo, cuadrado, diamante
series = [
    (v1, Vm1, sigma_Vm1, m1, C1, sm1, r21, "Serie 1"),
    (v2, Vm2, sigma_Vm2, m2, C2, sm2, r22, "Serie 2"),
    (v3, Vm3, sigma_Vm3, m3, C3, sm3, r23, "Serie 3"),
]

for (v, Vm, sVm, m, C, sm, r2, nombre), color, marker in zip(series, colores, marcadores):
    # Datos con barras
    plt.errorbar(v, Vm, yerr=sVm, fmt=marker, color=color, capsize=3,
        label=(f"{nombre}\n"
               f"h/e = {m:.2e} ± {sm:.1e} V·s\n"
               f"R² = {r2:.3f}"))
    # Ajuste
    x_fit = np.linspace(min(v), max(v), 200)
    plt.plot(x_fit, lin_func(x_fit, m, C), color=color)

plt.xlabel("Frecuencia v [Hz]")
plt.ylabel("Voltaje de frenado Vm [V]")
plt.title("Ajuste lineal de tres series de datos")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =======================
# comparación sin filtro - filtro - filtro pasco
# =======================

#orden 1 (sin filtro)
v1 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm1 = np.array([1.75, 1.54, 1.33, 0.91, 0.95])    # V ---
sigma_Vm1 = (0.5*Vm1)/100 + 0.01

#orden 1 (filtro)
v2 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm2 = np.array([1.800, 1.524, 1.319, 0.777, 0.601])    # V
sigma_Vm2 = (0.5*Vm2)/100 + 0.001

#orden 1 (filtro pasco)
v3 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm3 = np.array([1.798, 1.525, 1.315, 0.701, 0.601])    # V
sigma_Vm3 = (0.5*Vm3)/100 + 0.001

# =======================
# FUNCIÓN DE AJUSTE
# =======================
def lin_func(x, m, C):
    return m*x + C   # m = h/e

def ajustar(v, Vm, sigma_Vm):
    """Ajuste lineal con curve_fit y cálculo de R^2"""
    popt, pcov = curve_fit(lin_func, v, Vm, sigma=sigma_Vm,
                           absolute_sigma=True, p0=[4e-15, 0])
    m, C = popt
    sigma_m, sigma_C = np.sqrt(np.diag(pcov))
    Vm_pred = lin_func(v, *popt)
    ss_res = np.sum((Vm - Vm_pred)**2)
    ss_tot = np.sum((Vm - np.mean(Vm))**2)
    r2 = 1 - ss_res/ss_tot
    return m, C, sigma_m, sigma_C, r2

# =======================
# AJUSTES
# =======================
m1, C1, sm1, sC1, r21 = ajustar(v1, Vm1, sigma_Vm1)
m2, C2, sm2, sC2, r22 = ajustar(v2, Vm2, sigma_Vm2)
m3, C3, sm3, sC3, r23 = ajustar(v3, Vm3, sigma_Vm3)

# =======================
# RESULTADOS POR CONSOLA
# =======================
print("=== Sin filtro ===")
print(f"h/e = {m1:.3e} ± {sm1:.3e} V·s | C = {C1:.3e} ± {sC1:.3e} V | R² = {r21:.4f}")
print("\n=== Con filtro ===")
print(f"h/e = {m2:.3e} ± {sm2:.3e} V·s | C = {C2:.3e} ± {sC2:.3e} V | R² = {r22:.4f}")
print("\n=== Con filtro Pasco ===")
print(f"h/e = {m3:.3e} ± {sm3:.3e} V·s | C = {C3:.3e} ± {sC3:.3e} V | R² = {r23:.4f}")

# =======================
# GRÁFICO
# =======================
plt.figure(figsize=(8,6))

# Colores y símbolos diferentes
colores = [(0.6, 0.1, 0.1), (0.35, 0.0, 0.55), (0.50, 0.50, 0.0)]
marcadores = ['o', 's', 'D']  # círculo, cuadrado, diamante
series = [
    (v1, Vm1, sigma_Vm1, m1, C1, sm1, r21, "Sin filtros"),
    (v2, Vm2, sigma_Vm2, m2, C2, sm2, r22, "Con filtros a"),
    (v3, Vm3, sigma_Vm3, m3, C3, sm3, r23, "Con filtros b"),
]

for (v, Vm, sVm, m, C, sm, r2, nombre), color, marker in zip(series, colores, marcadores):
    # Datos con barras
    plt.errorbar(v, Vm, yerr=sVm, fmt=marker, color=color, capsize=3, markersize=3,
        label=(f"{nombre}\n"
               f"h/e = {m:.2e} ± {sm:.1e} V·s\n"))
               #f"R² = {r2:.3f}"
    # Ajuste
    x_fit = np.linspace(min(v), max(v), 200)
    plt.plot(x_fit, lin_func(x_fit, m, C), color=color)

plt.xlabel("Frecuencia v [Hz]")
plt.ylabel("Voltaje medido Vm [V]")
plt.title("")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig('comp3_orden1_tp3.png', dpi=300)
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =======================
# comparación sin filtro - filtro - filtro pasco
# =======================

#orden -1 (sin filtro)
v1 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm1 = np.array([1.71, 1.45, 1.30, 0.92, 0.97])    # V ---
sigma_Vm1 = (0.5*Vm1)/100 + 0.01

#orden -1 (filtro)
v2 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm2 = np.array([1.714, 1.464, 1.267, 0.742, 0.584])    # V
sigma_Vm2 = (0.5*Vm2)/100 + 0.001

#orden -1 (filtro pasco)
v3 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm3 = np.array([1.706, 1.471, 1.260, 0.776, 0.576])    # V
sigma_Vm3 = (0.5*Vm3)/100 + 0.001

# =======================
# FUNCIÓN DE AJUSTE
# =======================
def lin_func(x, m, C):
    return m*x + C   # m = h/e

def ajustar(v, Vm, sigma_Vm):
    """Ajuste lineal con curve_fit y cálculo de R^2"""
    popt, pcov = curve_fit(lin_func, v, Vm, sigma=sigma_Vm,
                           absolute_sigma=True, p0=[4e-15, 0])
    m, C = popt
    sigma_m, sigma_C = np.sqrt(np.diag(pcov))
    Vm_pred = lin_func(v, *popt)
    ss_res = np.sum((Vm - Vm_pred)**2)
    ss_tot = np.sum((Vm - np.mean(Vm))**2)
    r2 = 1 - ss_res/ss_tot
    return m, C, sigma_m, sigma_C, r2

# =======================
# AJUSTES
# =======================
m1, C1, sm1, sC1, r21 = ajustar(v1, Vm1, sigma_Vm1)
m2, C2, sm2, sC2, r22 = ajustar(v2, Vm2, sigma_Vm2)
m3, C3, sm3, sC3, r23 = ajustar(v3, Vm3, sigma_Vm3)

# =======================
# RESULTADOS POR CONSOLA
# =======================
print("=== Serie 1 ===")
print(f"h/e = {m1:.3e} ± {sm1:.3e} V·s | C = {C1:.3e} ± {sC1:.3e} V | R² = {r21:.4f}")
print("\n=== Serie 2 ===")
print(f"h/e = {m2:.3e} ± {sm2:.3e} V·s | C = {C2:.3e} ± {sC2:.3e} V | R² = {r22:.4f}")
print("\n=== Serie 3 ===")
print(f"h/e = {m3:.3e} ± {sm3:.3e} V·s | C = {C3:.3e} ± {sC3:.3e} V | R² = {r23:.4f}")

# =======================
# GRÁFICO
# =======================
plt.figure(figsize=(8,6))

# Colores y símbolos diferentes
colores = ['tab:blue', 'tab:red', 'tab:green']
marcadores = ['o', 's', 'D']  # círculo, cuadrado, diamante
series = [
    (v1, Vm1, sigma_Vm1, m1, C1, sm1, r21, "Serie 1"),
    (v2, Vm2, sigma_Vm2, m2, C2, sm2, r22, "Serie 2"),
    (v3, Vm3, sigma_Vm3, m3, C3, sm3, r23, "Serie 3"),
]

for (v, Vm, sVm, m, C, sm, r2, nombre), color, marker in zip(series, colores, marcadores):
    # Datos con barras
    plt.errorbar(v, Vm, yerr=sVm, fmt=marker, color=color, capsize=3,
        label=(f"{nombre}\n"
               f"h/e = {m:.2e} ± {sm:.1e} V·s\n"
               f"R² = {r2:.3f}"))
    # Ajuste
    x_fit = np.linspace(min(v), max(v), 200)
    plt.plot(x_fit, lin_func(x_fit, m, C), color=color)

plt.xlabel("Frecuencia v [Hz]")
plt.ylabel("Voltaje de frenado Vm [V]")
plt.title("Ajuste lineal de tres series de datos")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# =======================
# comparación sin filtro - filtro - filtro pasco
# =======================

#orden -2 (sin filtro)
v1 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm1 = np.array([1.67, 1.40, 1.25, 1.48, 1.16])    # V ---
sigma_Vm1 = (0.5*Vm1)/100 + 0.01

#orden -2 (filtro)
v2 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.18672e14]) #Hz
Vm2 = np.array([1.687, 1.408, 1.200, 0.513])    # V ---
sigma_Vm2 = (0.5*Vm2)/100 + 0.001

#orden -2 (filtro pasco)
v3 = np.array([8.20264e14, 7.40858e14, 6.87858e14, 5.48996e14, 5.18672e14]) #Hz
Vm3 = np.array([1.675, 1.404, 1.194, 0.594, 0.502])    # V
sigma_Vm3 = (0.5*Vm3)/100 + 0.001

# =======================
# FUNCIÓN DE AJUSTE
# =======================
def lin_func(x, m, C):
    return m*x + C   # m = h/e

def ajustar(v, Vm, sigma_Vm):
    """Ajuste lineal con curve_fit y cálculo de R^2"""
    popt, pcov = curve_fit(lin_func, v, Vm, sigma=sigma_Vm,
                           absolute_sigma=True, p0=[4e-15, 0])
    m, C = popt
    sigma_m, sigma_C = np.sqrt(np.diag(pcov))
    Vm_pred = lin_func(v, *popt)
    ss_res = np.sum((Vm - Vm_pred)**2)
    ss_tot = np.sum((Vm - np.mean(Vm))**2)
    r2 = 1 - ss_res/ss_tot
    return m, C, sigma_m, sigma_C, r2

# =======================
# AJUSTES
# =======================
m1, C1, sm1, sC1, r21 = ajustar(v1, Vm1, sigma_Vm1)
m2, C2, sm2, sC2, r22 = ajustar(v2, Vm2, sigma_Vm2)
m3, C3, sm3, sC3, r23 = ajustar(v3, Vm3, sigma_Vm3)

# =======================
# RESULTADOS POR CONSOLA
# =======================
print("=== Serie 1 ===")
print(f"h/e = {m1:.3e} ± {sm1:.3e} V·s | C = {C1:.3e} ± {sC1:.3e} V | R² = {r21:.4f}")
print("\n=== Serie 2 ===")
print(f"h/e = {m2:.3e} ± {sm2:.3e} V·s | C = {C2:.3e} ± {sC2:.3e} V | R² = {r22:.4f}")
print("\n=== Serie 3 ===")
print(f"h/e = {m3:.3e} ± {sm3:.3e} V·s | C = {C3:.3e} ± {sC3:.3e} V | R² = {r23:.4f}")

# =======================
# GRÁFICO
# =======================
plt.figure(figsize=(8,6))

# Colores y símbolos diferentes
colores = ['tab:blue', 'tab:red', 'tab:green']
marcadores = ['o', 's', 'D']  # círculo, cuadrado, diamante
series = [
    (v1, Vm1, sigma_Vm1, m1, C1, sm1, r21, "Serie 1"),
    (v2, Vm2, sigma_Vm2, m2, C2, sm2, r22, "Serie 2"),
    (v3, Vm3, sigma_Vm3, m3, C3, sm3, r23, "Serie 3"),
]

for (v, Vm, sVm, m, C, sm, r2, nombre), color, marker in zip(series, colores, marcadores):
    # Datos con barras
    plt.errorbar(v, Vm, yerr=sVm, fmt=marker, color=color, capsize=3,
        label=(f"{nombre}\n"
               f"h/e = {m:.2e} ± {sm:.1e} V·s\n"
               f"R² = {r2:.3f}"))
    # Ajuste
    x_fit = np.linspace(min(v), max(v), 200)
    plt.plot(x_fit, lin_func(x_fit, m, C), color=color)

plt.xlabel("Frecuencia v [Hz]")
plt.ylabel("Voltaje de frenado Vm [V]")
plt.title("Ajuste lineal de tres series de datos")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
he= np.array([3.938, 3.795, 4.023, 4.111, 3.821, 3.797, 4.023, 3.969])*10**-15
he_err= np.array([2.272, 2.371, 2.682, 2.670, 2.588, 2.584, 2.452, 2.557])*(10**-17)

C= np.array([-1.610, -1.573, -1.464, -1.539, -1.380, -1.361, -1.595, -1.543])
C_err= np.array([1.332, 1.406, 1.602, 1.588, 1.549, 1.547, 1., 1.561]) * 10**-2


def promedio_ponderado(valores, errores):
    valores = np.asarray(valores, dtype=float)
    errores = np.asarray(errores, dtype=float)
    pesos = 1 / errores**2
    promedio = np.sum(valores * pesos) / np.sum(pesos)
    error = np.sqrt(1 / np.sum(pesos))
    return promedio, error

prom_he, err_he = promedio_ponderado(he, he_err)
prom_C, err_C = promedio_ponderado(C, C_err)

print(f"he = {prom_he:.3e} ± {err_he:.3e}")
print(f"C  = {prom_C:.3f} ± {err_C:.3f}")
